# 非hot100外常考非MLDL代码题

## 快速排序&归并排序

In [11]:
def quick_sort(nums):
    if len(nums)<=1:
        return nums
    a = nums[len(nums)//2]  
    small = [x for x in nums if x<a]
    mid = [x for x in nums if x==a]
    big = [x for x in nums if x>a]
    return quick_sort(small)+mid+quick_sort(big)
arr = [5, 3, 8, 4, 2, 7, 1, 10]
sorted_arr = quick_sort(arr)
print(sorted_arr)
def sort(l,r):
    res = []
    i,j = 0,0
    while i < len(l) and j < len(r):
        if l[i] < r[j]:
            res.append(l[i])
            i += 1
        else:
            res.append(r[j])
            j += 1
    res.extend(l[i:])
    res.extend(r[j:])
    return res
def merge_sort(nums):
    if len(nums)<=1:
        return nums
    mid = len(nums)//2
    l = merge_sort(nums[:mid])
    r = merge_sort(nums[mid:])
    return sort(l,r)
arr = [5, 3, 8, 4, 2, 7, 1, 10]
sorted_arr = merge_sort(arr)
print(sorted_arr)


[1, 2, 3, 4, 5, 7, 8, 10]
[1, 2, 3, 4, 5, 7, 8, 10]


# ML&DL手撕题 （torch实现）

## Loss模块
softmax CE BCE
focalloss
hingeloss(SVM) tripletloss(锚点对比损失) infonceloss（对比损失）
BPRloss mse contrastiveloss
rqvae两部分loss recon+quantize

### softmax CE BCE
BCE（二元交叉熵）
配合Sigmoid激活函数。
CE（多元交叉熵）
配合Softmax激活函数。

In [76]:
#softmax函数和交叉熵损失
import torch
x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(x.max(dim=1))#返回的是value和index但是没有保留维度
print(x.max(dim=1, keepdim=True))#返回的是value和index并且保留维度
print(x.max(dim=1, keepdim=True)[0])#返回的是value并且保留维度
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))
print(sigmoid(torch.tensor([-1.0, 0.0, 1.0])))
def softmax(x):
    e = torch.exp(x - x.max(dim=1, keepdim=True)[0])#数值稳定的softmax
    return e / e.sum(dim=1, keepdim=True)
print(softmax(x))
def CE(pred, target):
    log_prob = torch.log(softmax(pred))#对预测值进行softmax并取对数
    loss = -log_prob[range(len(target)), target].mean()#计算交叉熵损失
    return loss
pred = torch.tensor([[2.0, 1.0, 0.1],[0.5, 1.5, 2.5]])
target = torch.tensor([0, 2])
print(CE(pred, target))
def BCE(pred, target):
    loss = -(target * torch.log(pred + 1e-15) + (1 - target) * torch.log(1 - pred + 1e-15)).mean()
    return loss
pred = torch.tensor([0.2, 0.8])
target = torch.tensor([0, 1])
print(BCE(pred, target))

torch.return_types.max(
values=tensor([3., 6.]),
indices=tensor([2, 2]))
torch.return_types.max(
values=tensor([[3.],
        [6.]]),
indices=tensor([[2],
        [2]]))
tensor([[3.],
        [6.]])
tensor([0.2689, 0.5000, 0.7311])
tensor([[0.0900, 0.2447, 0.6652],
        [0.0900, 0.2447, 0.6652]])
tensor(0.4123)
tensor(0.2231)


### Focal loss

In [ ]:
# Focal Loss是一种改进的交叉熵损失函数，主要用于处理类别不平衡，对难分类的样本给予更大的权重。
# alpha是平衡因子，alpha较大时模型关注更少数样本较小时则相反。
# gamma是调节因子，gamma越大，易分类样本的损失越小，难分类样本的损失越大。
import torch
import torch.nn.functional as F

def focalloss_multiclass(pred, target, alpha=1.0, gamma=0):
    p_t = F.softmax(pred, dim=-1)
    p_t = p_t[range(len(pred)), target]
    ce_loss = -torch.log(p_t + 1e-15)
    focal_loss = alpha * (1 - p_t) ** gamma * ce_loss
    return focal_loss.mean()
pred = torch.tensor([[0.5, 1.0, 1.5], [2.0, 2.5, 4.0]])  # logits
target = torch.tensor([0, 2], dtype=torch.long)
print(focalloss_multiclass(pred, target))

tensor([0.1863, 0.7361])
tensor([1.6803, 0.3064])
tensor([1.6803, 0.3064])
tensor(0.9933)


### 对比学习模块pointwise|pairwise|listwise|是RS中常见的任务lossfunction设计思路   Hingeloss Tripletloss InfoNCEloss
Pointwise Loss（点对损失）：对每个样本单独建模，常用回归或分类损失（如二分类交叉熵、均方误差）。
Pairwise Loss（对对损失）：对每个正负样本对建模，常用BPR Loss、Hinge Loss等。
 Listwise Loss（列表损失）：对一组样本（一个列表）整体建模，常用Softmax Cross Entropy。

In [ ]:
#Hinge Loss主要用于支持向量机（SVM）等分类模型，旨在最大化类别之间的间隔。
#对于正确分类的样本，损失为0；对于错误分类的样本，损失与预测值和真实标签之间的距离成正比。
#https://en.wikipedia.org/wiki/Hinge_loss
import torch
def hinge_loss(pred, target):
    loss = torch.clamp(1 - pred * target, min=0)
    return loss.mean()
pred = torch.tensor([0.5, -1.0, 1.5])#也就是y
target = torch.tensor([1, -1, 1])
print(hinge_loss(pred, target))
#Triplet Loss目的是把正样本之间拉近，把负样本之间推远。
#三个部分组成：anchor（锚点），positive（正样本）和negative（负样本）
#损失函数的目标是使得anchor与positive之间的距离小于anchor与negative之间的距离至少一个margin。
import torch
def triplet_loss(anchor, positive, negative, margin=1.0):
    pos_dist = torch.norm(anchor - positive, dim=1)
    neg_dist = torch.norm(anchor - negative, dim=1)
    loss = torch.relu(pos_dist - neg_dist + margin)
    return loss.mean()
positive = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
negative = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
anchor = torch.tensor([[1.5, 2.5], [3.5, 4.5]])
print(triplet_loss(anchor, positive, negative))
#INFONCE Loss 一般用于自监督学习（问：啥是自监督，有监督，无监督，半自监督o_O）
import torch
import torch.nn.functional as F
#一般是batch内的正样本和负样本,或者分成锚点和正负样本，如下所示
def info_nce_loss(query, pos, neg, t=0.07):
    """
    query: [batch_size, feature_dim]
    pos: [batch_size, feature_dim]
    neg: [batch_size, num_neg, feature_dim]
    t: 温度参数
    """
    # 归一化
    query = F.normalize(query, dim=-1)  # [B, D]
    pos = F.normalize(pos, dim=-1)      # [B, D]
    neg = F.normalize(neg, dim=-1)      # [B, N, D]
    # 计算相似度

    pos_sim = torch.sum(query * pos, dim=-1,keepdim = True)/t  # [B, 1]
    neg_sim = torch.matmul(query.unsqueeze(1), neg.transpose(1, 2)).squeeze(1)/t
    # 拼接正负样本
    logits = torch.cat([pos_sim, neg_sim], dim=1) # [B, 1+N]
    pos_exp = torch.exp(pos_sim)
    sum_exp = torch.exp(logits).sum(dim=1, keepdim=True)  # [B, 1]
    loss = -torch.log(pos_exp / sum_exp)
    return loss.mean()
query = torch.randn(4, 128)  # [B, D]
pos = torch.randn(4, 128)    # [B, D]
neg = torch.randn(4, 10, 128) # [B, N, D]
print(info_nce_loss(query, pos, neg))

tensor(0.1667)
tensor(0.)
tensor([[5.5228],
        [2.7679],
        [3.3137],
        [5.3177]])
tensor(4.2305)


### BPR loss RS必学！mse loss Contrastive loss

In [ ]:
#BPRloss （贝叶斯个性化排序损失）主要用于RS中的ranking，目的是最大化正样本和负样本之间的差异
#感觉问的确实比较多
import torch as t
torch.manual_seed(42)
#随机生成 3 个用户，每个用户对 6 个项目的评分向量（6维）
User = t.randn(3, 6, 6)
Poss = t.randn(3, 6, 6)
Neg = t.randn(3, 6, 6)
def inter(user,item):
    return (user * item).sum(dim=-1)
def pairloss(user,pos,neg):
    return -t.log(t.sigmoid(inter(user,pos) - inter(user,neg))).mean()
print(pairloss(User, Poss, Neg))
#MSE loss
import torch
def mse_loss(pred, target):
    return ((pred - target) ** 2).mean()
pred = torch.tensor([0.5, 0.8, 0.3])
target = torch.tensor([0.5, 0.7, 0.2])
print(mse_loss(pred, target))
#Contrastive loss
def contrastive_loss(x1, x2, label, m):
    dist = torch.norm(x1 - x2, p=2, dim=1)#实际上默认是欧式距离
    loss = label * dist**2 + (1 - label) * torch.clamp(m - dist, min=0)**2
    return loss.mean()
x1 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
x2 = torch.tensor([[1.5, 2.5], [5.0, 6.0]])
label = torch.tensor([1, 0])  # 1表示相似，0表示不相似
margin = 1.0
print(contrastive_loss(x1, x2, label, margin))


tensor(0.9805)
tensor(0.0067)
tensor(0.2500)


In [73]:
#我的项目中的RQVAEloss包括有重构损失，协同过滤损失，两部分的码本损失，同时对码本进行了一投影，这部分明天可以详细写一下。

## DL模块
dropout relu swish LN BN RMSNorm MHA mask 各种优化器 二维卷积 MLP

### Dropout relu swish LN BN RMSNorm（只除var不减mean）

In [ ]:
#DL模块
#用torch 
import torch
def dropout(x,p):
    mask = (torch.rand_like(x) > p).float()#float()是为了把bool类型转换成float类型，True变成1.0，False变成0.0😋
    return x * mask / (1 - p)
x = torch.tensor([[1.0, 2.0, 3.0],[4.0, 5.0, 6.0]])
print((torch.rand_like(x)>0.5).float())
print(dropout(x, 0.5))
def swish(x):
    return x * (1 / (1 + torch.exp(-x)))
def relu(x):
    return torch.clamp(x, min=0.0)
print(relu(torch.tensor([-1.0, 0.0, 1.0])))
#LN BN alpha gamma均是可学习参数
#https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html
def LN(x, gamma, alpha, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    #unbiased=True（默认）：计算无偏方差，即分母用 N-1；unbiased=False：计算有偏方差，即分母用N。
    normalized = (x - mean) / torch.sqrt(var + eps)
    return gamma * normalized + alpha
def BN(x, gamma, alpha, eps=1e-5):
    mean = x.mean(dim=0, keepdim=True)
    var = x.var(dim=0, keepdim=True, unbiased=False)
    normalized = (x - mean) / torch.sqrt(var + eps)
    return gamma * normalized + alpha
x = torch.tensor([[1.0, 2.0, 3.0],[4.0, 5.0, 6.0]])
gamma = 1
alpha = 0
print(LN(x, gamma, alpha))
print(BN(x, gamma, alpha))
def RMSNorm(alpha,beta,x):
    var = x.var(dim = -1, keepdim=True, unbiased=False)
    return alpha * x / torch.sqrt(var + 1e-8) + beta

tensor([[0., 0., 1.],
        [0., 1., 1.]])
tensor([[ 0.,  0.,  6.],
        [ 0.,  0., 12.]])
tensor([0., 0., 1.])
tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]])
tensor([[-1.0000, -1.0000, -1.0000],
        [ 1.0000,  1.0000,  1.0000]])


### MHA

In [ ]:
#https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
import torch
import torch.nn as nn
class MultiHeadAttention(nn.Module):
    def __init__(self,dim_in,dim_k,dim_v,num_heads = 8):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.dim_k = dim_k
        self.dim_v = dim_v
        self.W_q = nn.Linear(dim_in, dim_k * num_heads, bias=False)
        self.W_k = nn.Linear(dim_in, dim_k * num_heads, bias=False)
        self.W_v = nn.Linear(dim_in, dim_v * num_heads, bias=False)
        self.W_o = nn.Linear(dim_v * num_heads, dim_in, bias=False)
    def forward(self, x):
        batch_size, seq_len, dim_in = x.size()
        q = self.W_q(x).reshape(batch_size, seq_len, self.num_heads, self.dim_k).transpose(1, 2)
        k = self.W_k(x).reshape(batch_size, seq_len, self.num_heads, self.dim_k).transpose(1, 2)
        v = self.W_v(x).reshape(batch_size, seq_len, self.num_heads, self.dim_v).transpose(1, 2)
        #k的维度是(batch_size, num_heads, seq_len, dim_k)，v的维度是(batch_size, num_heads, seq_len, dim_v)
        #k.transpose(-2, -1)的维度是(batch_size, num_heads, dim_k, seq_len)，这样才能和q进行矩阵乘法
        #scores的维度是(batch_size, num_heads, seq_len, seq_len)
        #attn_weights的维度也是(batch_size, num_heads, seq_len, seq_len)
        #attn_output的维度是(batch_size, num_heads, seq_len, dim_v)
        #最后输出的维度是(batch_size, seq_len, dim_in)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.dim_k ** 0.5)
        attn_weights = torch.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v).transpose(1, 2).reshape(batch_size, seq_len, self.dim_v * self.num_heads)
        output = self.W_o(attn_output)
        return output
# 创建一个多头注意力层实例
dim_in = 16  # 输入维度
dim_k = 8    # 键和查询的维度
dim_v = 8    # 值的维度
num_heads = 4  # 头的数量
mha = MultiHeadAttention(dim_in, dim_k, dim_v, num_heads)
# 创建一个假设输入数据，形状 (batch_size, seq_len, dim_in)
batch_size = 2
seq_len = 5
x = torch.randn(batch_size, seq_len, dim_in)
output = mha(x)
print("Output shape:", output.shape)

Output shape: torch.Size([2, 5, 16])


### causalmask因果掩码 问：为什么要掩码 encoder decoder掩码方式区别～
一般来说causal和paddingmask只问过我方式

In [158]:
import torch
#triangular upper Matrix上三角矩阵
#triu上三角 tril下三角
def causal_mask(seq_len):
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
    mask[mask == 1] = float('-inf')
    return mask
print(causal_mask(5))

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])


import torch
def dropout(x):
    mask = (torch.rand_like(x)>p).float()
    return x*mask/(1-p)

In [ ]:
import torch
input_ids = torch.tensor([
    [1, 2, 3, 0, 0],    # 长度3，后面2个是padding
    [4, 5, 6, 7, 8],    # 长度5，无padding
    [9, 0, 0, 0, 0]     # 长度1，后面4个是padding
])
padding_mask = (input_ids == 0)
print(padding_mask)
# tensor([[False, False, False,  True,  True],
#         [False, False, False, False, False],
#         [False,  True,  True,  True,  True]])
attn_score = torch.randn(3, 5, 5)  # 假设的注意力分数
attn_score = attn_score.masked_fill(padding_mask.unsqueeze(1), float('-inf'))
print(attn_score)

tensor([[False, False, False,  True,  True],
        [False, False, False, False, False],
        [False,  True,  True,  True,  True]])
tensor([[[ 0.6664, -0.0743, -0.2096,    -inf,    -inf],
         [-0.9391, -0.6013, -0.0996,    -inf,    -inf],
         [-0.3313,  0.8436,  0.9874,    -inf,    -inf],
         [ 0.8244,  0.0247, -1.0641,    -inf,    -inf],
         [ 0.9624, -0.1426,  0.1527,    -inf,    -inf]],

        [[-1.5824,  0.9871,  1.1457, -0.1418, -0.2763],
         [-0.1932,  0.7768,  0.6839, -1.3246, -0.5161],
         [ 0.6002, -0.4702, -0.6086, -0.0462, -1.6457],
         [-0.4833, -0.7403,  0.3143,  0.1416,  1.0348],
         [-0.6264, -0.5151,  0.6903, -0.4940,  1.1366]],

        [[-0.4618,    -inf,    -inf,    -inf,    -inf],
         [ 1.0430,    -inf,    -inf,    -inf,    -inf],
         [ 1.1226,    -inf,    -inf,    -inf,    -inf],
         [-0.6837,    -inf,    -inf,    -inf,    -inf],
         [-0.4835,    -inf,    -inf,    -inf,    -inf]]])


### SGD Adagrad AdamW Adam RMSprop

In [ ]:
#Adam Adaptive Moment estimation 自适应优化器
#mt = beta1 * m(t-1) + (1 - beta1) * gt
#vt = beta2 * v(t-1) + (1 - beta2) * gt^2
#修偏： mt_hat = mt / (1 - beta1^t) vt_hat = vt / (1 - beta2^t)
#https://arxiv.org/pdf/1412.6980.pdf
import torch
class AdamOptimizer:
    def __init__(self,params,lr = 0.001,b1 = 0.9,b2 = 0.999,eps = 1e-8):
        self.params = list(params)
        self.lr = lr
        self.b1 = b1
        self.b2 = b2
        self.eps = eps
        
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        
        self.t = 0
    def step(self):
        self.t+=1
        for i,param in enumerate(self.params):
            grad = param.grad#获取当前的梯度
            self.m[i] = self.b1*self.m[i] + (1-self.b1)*grad
            self.v[i] = self.b2*self.v[i] + (1-self.b2)*grad**2
            
            m_hat = self.m[i]/(1-self.b1**self.t)
            v_hat = self.v[i]/(1-self.b2**self.t)
            
            param.data -= self.lr*m_hat/(torch.sqrt(v_hat)+self.eps)
    def zero_grad(self):
        for param in self.params:
            param.grad = None
torch.manual_seed(42)
# 创建两个可训练的张量，表示模型的参数
params = [torch.randn(3, 3, requires_grad=True),
          torch.randn(3, 1, requires_grad=True)]
optimizer = AdamOptimizer(params, lr=0.001)
loss = params[0].sum() + params[1].sum()  # 一个简单的损失函数，求参数的和
loss.backward()
optimizer.step()
optimizer.zero_grad()
# 参数 优化器 算loss 反向传播 更新参数 清零梯度

In [ ]:
import torch

class AdamWOptimizer:
    def __init__(self, params, lr=0.001, b1=0.9, b2=0.999, eps=1e-8, weight_decay=0.01):
        self.params = list(params)
        self.lr = lr
        self.b1 = b1
        self.b2 = b2
        self.eps = eps
        self.weight_decay = weight_decay
        
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        
        self.t = 0

    def step(self):
        self.t += 1
        for i, param in enumerate(self.params):
            grad = param.grad
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * grad
            self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * grad ** 2

            m_hat = self.m[i] / (1 - self.b1 ** self.t)
            v_hat = self.v[i] / (1 - self.b2 ** self.t)
            param.data -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps) + self.lr * self.weight_decay * param.data
    def zero_grad(self):
        for param in self.params:
            param.grad = None

torch.manual_seed(42)
params = [torch.randn(3, 3, requires_grad=True),
          torch.randn(3, 1, requires_grad=True)]
optimizer = AdamWOptimizer(params, lr=0.001, weight_decay=0.01)

loss = params[0].sum() + params[1].sum()
loss.backward()

optimizer.step()
optimizer.zero_grad()

# 查看更新后的参数
print(params[0])
print(params[1])

In [ ]:
import torch
class SGDOptimizer:
    def __init__(self,params,lr = 0.01):
        self.params = list(params)
        self.lr = lr
    def step(self):
        for param in self.params:
            param.data-=self.lr*param.grad
    def zero_grad(self):
        for param in self.params:
            param.grad = None
torch.manual_seed(42)
params = [torch.randn(3, 3, requires_grad=True),
            torch.randn(3, 1, requires_grad=True)]  
optimizer = SGDOptimizer(params, lr=0.01)
loss = params[0].sum() + params[1].sum()
loss.backward()
optimizer.step()
optimizer.zero_grad()
print(params[0])
print(params[1])

tensor([[ 0.3267,  0.1188,  0.2245],
        [ 0.2203, -1.1329, -0.1963],
        [ 2.1982, -0.6480,  0.4517]], requires_grad=True)
tensor([[0.2574],
        [0.5249],
        [0.7994]], requires_grad=True)


In [ ]:
#写多了就会发现 SGD Adagrad AdamW Adam RMSprop都大同小异
import torch
class Adagrad:
    def __init__(self,params,lr,eps=1e-10):
        self.params = list(params)
        self.lr = lr
        self.eps = eps
        self.v = [torch.zeros_like(p) for p in self.params]
    def step(self):
        for i,param in enumerate(self.params):
            grad = param.grad
            self.v[i] += grad ** 2
            param.data -= self.lr * grad / (torch.sqrt(self.v[i]) + self.eps)
    def zero_grad(self):
        for param in self.params:
            param.grad = None
torch.manual_seed(42)
params = [torch.randn(3, 3, requires_grad=True),
          torch.randn(3, 1, requires_grad=True)]
optimizer = Adagrad(params, lr=0.01)
loss = params[0].sum() + params[1].sum()
loss.backward()
optimizer.step()
optimizer.zero_grad()
print(params[0])
print(params[1])

tensor([[ 0.3267,  0.1188,  0.2245],
        [ 0.2203, -1.1329, -0.1963],
        [ 2.1982, -0.6480,  0.4517]], requires_grad=True)
tensor([[0.2574],
        [0.5249],
        [0.7994]], requires_grad=True)


### 2维卷积

In [ ]:
import torch
def conv2d(input, kernel):
    H,W = input.shape
    kH,kW = kernel.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    output = torch.zeros(out_H, out_W)
    for i in range(out_H):
        for j in range(out_W):
            output[i, j] = (input[i:i+kH, j:j+kW] * kernel).sum()
    return output
input = torch.tensor([[1.0, 2.0, 3.0],
                      [4.0, 5.0, 6.0],
                      [7.0, 8.0, 9.0]])
kernel = torch.tensor([[0.0, 1.0],
                       [1.0, 0.0]])
print(conv2d(input, kernel))

tensor([[ 6.,  8.],
        [12., 14.]])


### MLP（CE+backward+relu+softmax）

In [ ]:
def softmax(x):
    e = torch.exp(x-x.max(dim = 1,keepdim=True)[0])
    return e / e.sum(dim=1,keepdim=True)
def relu(x):
    return torch.maximum(x, torch.zeros_like(x))
class MLP:
    def __init__(self,input_size,hidden_size,output_size):
        self.W1 = torch.randn(input_size,hidden_size)*0.01
        self.b1 = torch.zeros(1,hidden_size)
        self.W2 = torch.randn(hidden_size,output_size)*0.01
        self.b2 = torch.zeros(1,output_size)

In [ ]:
import numpy as np
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)
def relu(x):
    return np.maximum(x, 0)
class MLP:
    def __init__(self,i_s,h_s,o_s):
        self.w1 = np.random.randn(i_s,h_s)
        self.b1 = np.random.randn(1,h_s)
        self.w2 = np.random.randn(h_s,o_s)
        self.b2 = np.random.randn(1,o_s)

    def forward(self,x):
        self.z1 = np.dot(x,self.w1)+self.b1
        self.a1 = relu(self.z1)
        self.z2 = np.dot(self.a1,self.w2)+self.b2
        self.a2 = softmax(self.z2)
        return self.a2
    def backward(self,y_pred,y_true,lr):
        m = y_true.shape[0]
        dA2 = y_pred - y_true #bs_os
        dW2 = np.dot(self.a1.T, dA2)/m #hs_os  
        db2 = np.sum(dA2,axis = 0,keepdim = True)/m #1_os
        
        dA1 = np.dot(dA2,self.w2.T) #bs_hs
        dZ1 = dA1 * (self.z1 > 0) #bs_hs
        dW1 = np.dot(self.x.T,dZ1)/m
        db1 = np.sum(dA1,axis = 0, keepdim = True)/m
        self.w1 -= lr * dW1
        self.b1 -= lr * db1
        self.w2 -= lr * dW2
        self.b2 -= lr * db2
        
        
        
        

## ML 模块
kmeans kmeans++ LR LR

### Kmeans and Kmeans_plus

In [207]:
import torch
def kmeans(x,k,iter = 100):
    torch.manual_seed(42)
    n = x.shape[0]
    c = x[torch.randperm(n)[:k]]#生成一个0到n-1的随机排列，取前k个作为初始中心
    for i in range(iter):
        d = torch.cdist(x,c)#计算每个点到每个中心的距离
        l = torch.argmin(d,dim=1)#每个点分配到最近的中心
        c_ = torch.stack([x[l==j].mean(0) for j in range(k)])
        if torch.norm(c-c_) < 1e-5:
            break
        c = c_
    return c,l
x = torch.randn(100,2)
c,l = kmeans(x,10)
print(c)
def kmeansplus(x,k,iter = 100):
    torch.manual_seed(42)
    n = x.shape[0]
    first_idx = torch.randint(n,(1,))#随机选择第一个中心
    c = [x[first_idx]]
    for _ in range(1,k):
        c_tensor = torch.cat(c, dim=0)

        d = torch.cdist(x,c_tensor)#计算每个点到当前中心的距离
        p = d.min(1)[0]**2#距离的平方作为概率
        p /= p.sum()#归一化概率
        
        c.append(x[torch.multinomial(p, 1)])#根据概率选择下一个中心
    c = torch.cat(c,dim = 0)
    for i in range(iter):
        d = torch.cdist(x,c)
        l = torch.argmin(d,dim=1)
        c_ = torch.stack([x[l==j].mean(0) for j in range(k)])
        if torch.norm(c-c_) < 1e-5:
            break
        c = c_
    return c,l
x = torch.randn(100,2)
c,l = kmeansplus(x,10)
print(c)

tensor([[ 0.1024, -1.4960],
        [-1.3257, -0.2296],
        [ 0.8559, -0.4050],
        [-0.7327,  1.1278],
        [ 1.4961,  0.4438],
        [ 1.2632,  3.3657],
        [ 1.4660, -1.3017],
        [-0.2501,  0.4768],
        [ 0.3207,  1.1832],
        [-0.4002, -0.7275]])
tensor([[-0.6904,  0.9859],
        [ 0.6098,  0.7237],
        [-1.6535, -0.7003],
        [ 0.6856, -0.4346],
        [-0.6049, -0.0294],
        [ 0.9009, -1.8033],
        [-2.3184,  1.3032],
        [-0.2908, -1.2381],
        [ 1.6817,  0.3484],
        [ 0.0549,  0.5853]])


### linear regression & logistic regression

In [203]:
import torch
def linear_regression(X,y,lr = 0.03,num_epochs = 100):
    num_feature = X.shape[1]
    num_samples = x.shape[0]
    w = torch.randn(num_feature)
    b = torch.randn(1)
    for epoch in range(num_epochs):
        y_pred = X@w + b
        # loss = ((y_pred-y)**2).mean()
        grad_w = (2/num_samples)*(X.T@(y_pred-y))
        grad_b = (2/num_samples)*(y_pred-y).sum()
        w -= lr*grad_w
        b -= lr*grad_b
    return w,b
def sigmoid(x):
    return 1/(1+torch.exp(-x))
def logistic_regression(X,y,lr = 0.03,num_epochs = 100):
    num_feature = X.shape[1]
    num_samples = x.shape[0]
    w = torch.randn(num_feature)
    b = torch.randn(1)
    for epoch in range(num_epochs):
        Z = X@w + b
        y_pred = sigmoid(Z)
        # loss = y*(torch.log(y_pred+1e-8))+(1-y)*(torch.log((1-y_pred)+1e-8))
        grad_w = (1 / num_samples) * (X.T @ (y_pred - y))
        grad_b = (1 / num_samples) * (y_pred - y).sum()
        w -= lr*grad_w
        b -= lr*grad_b
    return w,b
# 生成简单数据：y = 2*x1 - 3*x2 + 1
torch.manual_seed(0)
X = torch.randn(100, 2)
true_w = torch.tensor([2.0, -3.0])
true_b = 1.0
y = X @ true_w + true_b + torch.randn(100) * 0.1  # 加一点噪声

w, b = linear_regression(X, y, lr=0.05, num_epochs=200)
print("线性回归结果：")
print("Learned w:", w)
print("Learned b:", b)
print("True w:", true_w)
print("True b:", true_b)
# 生成简单二分类数据：y = sigmoid(2*x1 - 3*x2 + 1) > 0.5
torch.manual_seed(0)
X = torch.randn(100, 2)
true_w = torch.tensor([2.0, -3.0])
true_b = 1.0
logits = X @ true_w + true_b
y = (sigmoid(logits) > 0.5).float()  # 标签为0或1

w, b = logistic_regression(X, y, lr=0.1, num_epochs=200)
print("\n逻辑回归结果：")
print("Learned w:", w)
print("Learned b:", b)
print("True w:", true_w)
print("True b:", true_b)

线性回归结果：
Learned w: tensor([ 2.0077, -2.9875])
Learned b: tensor([1.0015])
True w: tensor([ 2., -3.])
True b: 1.0

逻辑回归结果：
Learned w: tensor([ 1.3341, -2.2175])
Learned b: tensor([0.9065])
True w: tensor([ 2., -3.])
True b: 1.0


## 模型评价指标
AUC GAUC Accuracy precision recall f1socre topk topp

### AUC｜GAUC 

In [204]:
def auc_score(y_true, y_pred):
    """
    y_true: 真实标签（0或1），形状 [n]
    y_pred: 预测概率，形状 [n]
    """
    pos = []
    neg = []
    for yt, yp in zip(y_true, y_pred):
        if yt == 1:
            pos.append(yp)
        else:
            neg.append(yp)
    num = 0
    total = 0
    for p in pos:
        for n in neg:
            if p > n:
                num += 1
            elif p == n:
                num += 0.5
            total += 1
    return num / total if total > 0 else 0
y_true = [0, 1, 0, 1, 1, 0]
y_pred = [0.1, 0.8, 0.3, 0.7, 0.6, 0.2]
print("AUC:", auc_score(y_true, y_pred))

from collections import defaultdict
def gauc_score(group_ids,y_true,p_pred):
    group_data = defaultdict(list)
    for gid,yt,yp in zip(group_ids,y_true,y_pred):
        group_data[gid].append((yt,yp))
    total_pairs = 0
    weighted_auc = 0
    for gid,samples in group_data.items():
        y_g = [s[0] for s in samples]
        y_p = [s[1] for s in samples]
        pos_num = sum(y_g)
        neg_num = len(y_g)-pos_num
        pair_num = pos_num*neg_num
        if pair_num == 0:
            continue
        auc = auc_score(y_g,y_p)
        weighted_auc += auc * pair_num
        total_pairs += pair_num
        return weighted_auc / total_pairs if total_pairs > 0 else 0
group_ids = ['A', 'A', 'B', 'B', 'B', 'C']
y_true =    [0,   1,   0,   1,   1,   0]
y_pred =    [0.1, 0.8, 0.3, 0.7, 0.6, 0.2]
print("GAUC:", gauc_score(group_ids, y_true, y_pred))

AUC: 1.0
GAUC: 1.0


### Accuracy|Precision|Recall|F1_score

In [205]:
def accuracy(y_true, y_pred):
    correct = sum([yt == yp for yt, yp in zip(y_true, y_pred)])
    return correct / len(y_true)
def precision(y_true, y_pred):
    tp = sum([yt == 1 and yp == 1 for yt, yp in zip(y_true, y_pred)])
    fp = sum([yt == 0 and yp == 1 for yt, yp in zip(y_true, y_pred)])
    return tp / (tp + fp) if (tp + fp) > 0 else 0
def recall(y_true, y_pred):
    tp = sum([yt == 1 and yp == 1 for yt, yp in zip(y_true, y_pred)])
    fn = sum([yt == 1 and yp == 0 for yt, yp in zip(y_true, y_pred)])
    return tp / (tp + fn) if (tp + fn) > 0 else 0
def f1_score(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0
y_true = [0, 1, 1, 0, 1, 0, 1, 1]
y_pred = [0, 1, 0, 0, 1, 1, 1, 1]

print("Accuracy:", accuracy(y_true, y_pred))
print("Precision:", precision(y_true, y_pred))
print("Recall:", recall(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))

Accuracy: 0.75
Precision: 0.8
Recall: 0.8
F1-score: 0.8000000000000002


### Top k / p

In [206]:
def top_k_accuracy(y_true, y_pred_scores, k=5):
    """
    y_true: 真实类别索引（如[2, 0, 1]）
    y_pred_scores: 每个样本的预测分数（如[[0.1,0.7,0.2], ...]）
    k: top-k值
    """
    correct = 0
    for yt, scores in zip(y_true, y_pred_scores):
        topk = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        if yt in topk:
            correct += 1
    return correct / len(y_true)
y_true = [2, 0, 1]
y_pred_scores = [
    [0.1, 0.7, 0.2],  # 真实类别2，分数最高的是1、2、0
    [0.8, 0.1, 0.1],  # 真实类别0，分数最高的是0、1、2
    [0.3, 0.5, 0.2]   # 真实类别1，分数最高的是1、0、2
]
print("Top-2 Accuracy:", top_k_accuracy(y_true, y_pred_scores, k=2))

def top_p_accuracy(y_true, y_pred_scores, p=0.8):
    """
    y_true: 真实类别索引
    y_pred_scores: 每个样本的预测分数（如概率）
    p: top-p阈值
    """
    correct = 0
    for yt, scores in zip(y_true, y_pred_scores):
        # 排序
        sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        sorted_scores = [scores[i] for i in sorted_indices]
        cum_prob = 0
        top_p_indices = []
        for idx, prob in zip(sorted_indices, sorted_scores):
            cum_prob += prob
            top_p_indices.append(idx)
            if cum_prob >= p:
                break
        if yt in top_p_indices:
            correct += 1
    return correct / len(y_true)
y_true = [2, 0, 1]
y_pred_scores = [
    [0.1, 0.7, 0.2],  # 真实类别2
    [0.8, 0.1, 0.1],  # 真实类别0
    [0.3, 0.5, 0.2]   # 真实类别1
]
print("Top-p(0.8) Accuracy:", top_p_accuracy(y_true, y_pred_scores, p=0.8))

Top-2 Accuracy: 1.0
Top-p(0.8) Accuracy: 1.0
